# Demo del pipeline RAG de auditoría financiera (EP1 - ISY0101)

Este notebook ejecuta el flujo completo de punta a punta:
1. Ingesta y chunking de `data/raw/`.
2. Construcción del índice FAISS.
3. Consulta al agente de auditoría (retriever -> LLM-judge -> prompt -> Gemini -> Pydantic).
4. Verificación fáctica de la respuesta (fact-checking).
5. Cálculo de las 4 métricas RAG y del resumen de costo/latencia.

**Requisito:** definir `GOOGLE_API_KEY` en un archivo `.env` en la raíz del
proyecto (ver `.env.example`) antes de ejecutar las celdas que llaman al LLM.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.loaders import ingest_and_index
from src.rag_pipeline import AuditRagPipeline
from src.fact_checking import verify_answer
from src.metrics import evaluate_rag_response

## 1-2. Ingesta + índice FAISS

In [ ]:
vectorstore = ingest_and_index()
print(f"Vectores indexados: {vectorstore.index.ntotal}")

## 3. Consulta al agente

In [ ]:
pipeline = AuditRagPipeline()
resultado = pipeline.run(
    "¿Existen señales de riesgo de fraude en la nota de crédito NC-045?"
)
print(resultado.informe.model_dump_json(indent=2))
print("Chunks recuperados:", resultado.retrieved_chunk_ids)
print("Chunks retenidos por el juez:", resultado.kept_chunk_ids)
print("Métricas de observabilidad:", resultado.callback.summary())

## 4. Fact-checking de la respuesta

In [ ]:
reporte_fc = verify_answer(
    answer=resultado.informe.model_dump_json(),
    context=resultado.context_text,
)
for claim in reporte_fc.afirmaciones:
    print(f"[{claim.veredicto}] {claim.afirmacion} -> {claim.evidencia}")

## 5. Métricas RAG (context_precision, context_recall, faithfulness, answer_relevancy)

In [ ]:
relevant_ids = {0, 1, 2}  # ground truth de ejemplo: ajustar según el caso real

eval_result = evaluate_rag_response(
    retrieved_ids=resultado.retrieved_chunk_ids,
    relevant_ids=relevant_ids,
    claims=reporte_fc.afirmaciones,
    answer=resultado.informe.model_dump_json(),
    question="¿Existen señales de riesgo de fraude en la nota de crédito NC-045?",
)
eval_result.as_dict()